# DisputeCourt — GRPO on free Colab (T4)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

This notebook deliberately does **not** install `trl`. GRPO is implemented
directly in `training/grpo_minimal.py` against plain torch + transformers +
peft, because the TRL/torchao/peft version matrix on Colab breaks often and
debugging it is not a good use of a deadline.

End to end: ~25-35 min on a free T4.


## 1. Check the GPU


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > T4 GPU'


## 2. Install

Only `peft`. torch and transformers already ship with Colab, and *not*
touching them is what keeps this reproducible.


In [ ]:
# Colab ships torchao 0.10.0. peft's LoRA dispatcher calls
# is_torchao_available(), which RAISES (rather than returning False) on an
# incompatible torchao version, so get_peft_model() dies before it ever
# reaches our layers. We never want the torchao path -- removing the package
# makes that check return False and peft skips it.
!pip uninstall -y -q torchao
!pip install -q 'peft>=0.11.0'

import peft, transformers, torch
print('peft', peft.__version__, '| transformers', transformers.__version__,
      '| torch', torch.__version__)

# Fail here, loudly, rather than 200 GRPO steps later.
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
_m = AutoModelForCausalLM.from_pretrained('hf-internal-testing/tiny-random-gpt2')
_m = get_peft_model(_m, LoraConfig(task_type='CAUSAL_LM', target_modules=['c_attn']))
print('LoRA injection works')
del _m


## 3. Get the code


In [ ]:
# Clone if absent, then HARD RESET to origin/main.
# The previous version was `git clone ... || (cd ... && git pull -q)`, where
# -q swallowed the pull's failure: on a re-run the clone failed (directory
# exists), the pull silently did nothing, and the session kept running stale
# code while looking like it had updated. Fetch + reset is unambiguous.
import os
REPO = 'https://github.com/AniketAslaliya/disputecourt.git'
if not os.path.exists('/content/disputecourt'):
    !git clone -q {REPO} /content/disputecourt
%cd /content/disputecourt
!git fetch origin && git reset --hard origin/main

# Print the commit you are actually running. If this is older than the
# fix you expect, nothing below is testing what you think it is.
!git log --oneline -1


## 4. Baseline: the base model, before any RL

Same prompt, same eval split, same parser as the tuned run below. The only
difference between the two is the LoRA adapter, which is what makes the
delta attributable to GRPO rather than to harness changes.


In [ ]:
!python eval/run_model_eval.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --out data/results_base.jsonl \
    --label 'Base Qwen2.5-0.5B (no RL)'


## 5. GRPO

`--steps` is the number of prompts (one group of `--group-size` completions
each). 250 x 6 is roughly 20 min on a T4. Watch `reward(last20)` in the log:
if it climbs, the policy is learning; if it flatlines while
`skipped (zero-variance groups)` grows, the policy has collapsed onto one
verdict and the run is telling you so.


In [ ]:
!python training/grpo_minimal.py \n    --steps 250 \n    --group-size 6 \n    --micro-batch 2 \n    --max-new-tokens 128 \n    --lr 1e-5 \n    --kl-beta 0.02 \n    --accum 2 \n    --output training/checkpoints

# If this OOMs, drop --micro-batch to 1. It changes peak memory only,
# not the gradient: partial losses share one group-wide normaliser, so
# the accumulated gradient is identical either way.


## 6. Evaluate the tuned policy


In [ ]:
!python eval/run_model_eval.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --adapter training/checkpoints \
    --out data/results_grpo.jsonl \
    --label 'GRPO-tuned'


## 7. The table

Keyword control vs base model vs GRPO-tuned, on the same 100 held-out cases.


In [ ]:
!python eval/keyword_baseline.py
!python eval/compare_all.py


## 8. Training curve


In [ ]:
import json, os
import matplotlib.pyplot as plt

HIST = 'training/checkpoints/train_history.json'

# This file is written at the END of grpo_minimal.py. If it is missing, the
# training cell did not finish -- plotting is not the problem, and a bare
# FileNotFoundError here hides which cell actually failed.
if not os.path.exists(HIST):
    print('=' * 68)
    print('NO TRAINING HISTORY -- the GRPO cell did not complete.')
    print('=' * 68)
    print('This cell is fine. Scroll UP to the GRPO training cell and read its\n'
          'output -- that is where the real error is.\n')
    print('Checks, in order:')
    os.system('git log --oneline -1')
    print('  ^ must be c4d5a92 or newer, else you are running stale code')
    print()
    os.system('ls -la training/checkpoints/ 2>&1 | head -5')
    print()
    os.system('nvidia-smi --query-gpu=memory.used,memory.total --format=csv')
    print('\nIf the GRPO cell showed CUDA OutOfMemory, re-run it with '
          '--micro-batch 1.')
else:
    h = json.load(open(HIST))
    scored = [x for x in h if not x['skipped']]
    if not scored:
        print(f'{len(h)} groups recorded but every one was skipped as '
              'zero-variance -- the policy gave identical rewards across each '
              'group, so there was no gradient signal. That is policy collapse, '
              'and it is a reportable result, not a plotting bug.')
    else:
        r = [x['reward_mean'] for x in scored]
        w = 20
        smooth = [sum(r[max(0, i - w):i + 1]) / len(r[max(0, i - w):i + 1])
                  for i in range(len(r))]
        plt.figure(figsize=(9, 4))
        plt.plot(r, alpha=0.25, label='per-group mean reward')
        plt.plot(smooth, lw=2, label=f'{w}-group moving average')
        plt.xlabel('GRPO step'); plt.ylabel('reward'); plt.legend()
        plt.title('DisputeCourt GRPO training reward'); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig('data/training_curve.png', dpi=140)
        plt.show()
        n = max(1, len(r) // 4)
        print(f'mean reward, first quarter: {sum(r[:n])/n:+.3f}')
        print(f'mean reward, last quarter:  {sum(r[-n:])/n:+.3f}')
        print('skipped zero-variance groups:',
              sum(1 for x in h if x['skipped']), '/', len(h))


## 9. Download the results

Pull these four files down and commit them to the repo -- they are the
evidence behind the README's metrics table.


In [ ]:
from google.colab import files
for f in ['data/results_base.jsonl', 'data/results_grpo.jsonl',
          'data/results_base.summary.json', 'data/results_grpo.summary.json',
          'data/comparison.json', 'data/training_curve.png']:
    try:
        files.download(f)
    except Exception as e:
        print('skip', f, e)
